In [7]:
# Install VibeVoice
!pip install -q vibevoice soundfile

In [ ]:
# Patch VibeVoice: add None-guards for speech_tensors/speech_masks in the prefill block.
# Uses find_spec (not import) to locate the file so the module isn't cached before patching.
import importlib.util, pathlib

_spec = importlib.util.find_spec("vibevoice.modular.modeling_vibevoice_inference")
_path = pathlib.Path(_spec.origin)
_src = _path.read_text()

_patches = {
    '"speech_tensors": speech_tensors.to(device=device),':
        '"speech_tensors": speech_tensors.to(device=device) if speech_tensors is not None else None,',
    '"speech_masks": speech_masks.to(device),':
        '"speech_masks": speech_masks.to(device) if speech_masks is not None else None,',
    '"speech_input_mask": speech_input_mask.to(device),':
        '"speech_input_mask": speech_input_mask.to(device) if speech_input_mask is not None else None,',
}

for old, new in _patches.items():
    if old in _src:
        _src = _src.replace(old, new)
        print(f"Patched: {old.strip()}")
    elif new in _src:
        print(f"Already patched: {old.strip()}")
    else:
        print(f"NOT FOUND — version mismatch: {old.strip()}")

_path.write_text(_src)
print("File written. Run the cells below fresh (no restart needed).")

In [9]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch Runtime to T4")

GPU: Tesla T4


In [10]:
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

MODEL_ID = "vibevoice/VibeVoice-1.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} on {device}...")
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(device)
model.eval()
print("Model ready.")

Loading vibevoice/VibeVoice-1.5B on cuda...


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'Qwen2Tokenizer'. 
The class this function is called from is 'VibeVoiceTextTokenizerFast'.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model ready.


In [11]:
import re

# Paste your script here, or replace with: open("your_script.txt").read()
RAW_SCRIPT = """\
Maya: Kate says she's saving herself—classic.
Ken: Well actually, Lost treats her "escape mindset" like the Island has a billing department.
Maya: You think you're leaving and the Island's like, Cute. Anyway.
Ken: Hey, Maya. Welcome back to Lost Deep Dive. Today we're doing a Kate-focused dive into how Lost uses her motives—escape, control, protection, guilt—to build mythology.
Maya: And I'm already bracing for the part where we're told she's just reacting.
"""

def remap_speakers(text: str) -> str:
    """Convert 'Name: dialogue' lines to 'Speaker N: dialogue' (0-indexed)."""
    speaker_map: dict = {}
    out = []
    for line in text.splitlines():
        m = re.match(r"^([A-Za-z][A-Za-z0-9 _-]*):\s*(.+)$", line.strip())
        if m:
            name, dialogue = m.group(1).strip(), m.group(2)
            if name not in speaker_map:
                speaker_map[name] = len(speaker_map)
            out.append(f"Speaker {speaker_map[name]}: {dialogue}")
        elif line.strip():
            out.append(line.strip())
    print("Speaker mapping:", speaker_map)
    return "\n".join(out)

script = remap_speakers(RAW_SCRIPT)
print(script)

Speaker mapping: {'Maya': 0, 'Ken': 1}
Speaker 0: Kate says she's saving herself—classic.
Speaker 1: Well actually, Lost treats her "escape mindset" like the Island has a billing department.
Speaker 0: You think you're leaving and the Island's like, Cute. Anyway.
Speaker 1: Hey, Maya. Welcome back to Lost Deep Dive. Today we're doing a Kate-focused dive into how Lost uses her motives—escape, control, protection, guilt—to build mythology.
Speaker 0: And I'm already bracing for the part where we're told she's just reacting.


In [13]:
inputs = processor(text=script, return_tensors="pt", padding=True)
inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

print("Generating audio...")
with torch.inference_mode():
    output = model.generate(
        **inputs,
        tokenizer=processor.tokenizer,
        return_speech=True,
    )
print("Done.")

Generating audio...


AttributeError: 'NoneType' object has no attribute 'to'

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

SAMPLE_RATE = 24000
OUT_FILE = "episode_clip.wav"

chunks = [c for c in output.speech_outputs if c is not None]
audio = torch.cat(chunks, dim=-1).squeeze().cpu().float().numpy()

sf.write(OUT_FILE, audio, SAMPLE_RATE)
print(f"Saved {len(audio)/SAMPLE_RATE:.1f}s → {OUT_FILE}")
display(Audio(audio, rate=SAMPLE_RATE))